# 🧬 Self-Replicating Coding Agent — Colab Backend

Runs the full evolution pipeline locally on T4 GPU using **Qwen3-14B** via Ollama.
No Groq/Gemini rate limits. Live web dashboard served via ngrok.

| | |
|---|---|
| **Hardware** | T4 GPU (~2 Colab units/hr) |
| **Credits** | 300 units → **~150 hours** of evolution |
| **Model** | `qwen3:14b` — ~9 GB VRAM, built-in reasoning mode |
| **Dashboard** | Live MISSION_CONTROL via ngrok public URL |

---
### Run order
**Cell 8** (keep-alive) → **Cell 1** → **Cell 2** → **Cell 3** *(~5 min first time)* → **Cell 4** → **Cell 5** → **Cell 9** *(get dashboard URL)* → **Cell 6** 🚀

In [ ]:
#@title ⏰ Cell 8 — Keep-alive (run FIRST)
#@markdown Prevents Colab idle timeout. Run this before everything else.

import threading, time, datetime

_keep_alive = True

def _heartbeat():
    start = datetime.datetime.now()
    while _keep_alive:
        elapsed = datetime.datetime.now() - start
        h, r = divmod(int(elapsed.total_seconds()), 3600)
        m, s = divmod(r, 60)
        print(f'💓 {datetime.datetime.now().strftime("%H:%M:%S")} — running {h:02d}h {m:02d}m', end='\r')
        time.sleep(30)

threading.Thread(target=_heartbeat, daemon=True).start()
print('✅ Keep-alive started')
print('   To stop: _keep_alive = False')

In [ ]:
#@title 📂 Cell 1 — Mount Google Drive & clone project

from google.colab import drive
import os, shutil, subprocess

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/SelfReplicatingAgent'
WORK_DIR  = '/content/SelfReplicatingAgent'
os.makedirs(DRIVE_DIR, exist_ok=True)

zip_path = f'{DRIVE_DIR}/2ndRunSelfReplicatingAgent.zip'
if os.path.exists(zip_path) and not os.path.exists(WORK_DIR):
    print('📦 Extracting zip...')
    shutil.unpack_archive(zip_path, '/content/')
    extracted = [d for d in os.listdir('/content/')
                 if 'SelfReplicating' in d and os.path.isdir(f'/content/{d}')]
    if extracted and f'/content/{extracted[0]}' != WORK_DIR:
        os.rename(f'/content/{extracted[0]}', WORK_DIR)
    print('✅ Extracted')
elif not os.path.exists(WORK_DIR):
    print('🔗 Cloning from HuggingFace...')
    subprocess.run(['git', 'clone', '--depth=1',
        'https://huggingface.co/spaces/Balaji33k/self-replicating-agent', WORK_DIR], check=True)
    print('✅ Cloned')
else:
    print(f'✅ Already at {WORK_DIR}')

# Persist data/ to Drive
DATA_DRIVE, DATA_LOCAL = f'{DRIVE_DIR}/data', f'{WORK_DIR}/data'
os.makedirs(DATA_DRIVE, exist_ok=True)
if os.path.exists(DATA_LOCAL) and not os.path.islink(DATA_LOCAL):
    for f in os.listdir(DATA_LOCAL):
        dst = f'{DATA_DRIVE}/{f}'
        if not os.path.exists(dst): shutil.copy2(f'{DATA_LOCAL}/{f}', dst)
    shutil.rmtree(DATA_LOCAL)
if not os.path.islink(DATA_LOCAL):
    os.symlink(DATA_DRIVE, DATA_LOCAL)
    print('🔗 data/ → Drive (results persist)')

print('\n✅ Ready:', os.listdir(WORK_DIR))

In [ ]:
#@title 📦 Cell 2 — Install dependencies

import subprocess, sys
WORK_DIR = '/content/SelfReplicatingAgent'

r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{WORK_DIR}/requirements.txt'],
    capture_output=True, text=True
)
print('✅ Installed' if r.returncode == 0 else f'❌ {r.stderr[-1000:]}')

# Also install pyngrok for dashboard
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], capture_output=True)
print('✅ pyngrok installed')

for pkg in ['langchain_groq', 'langchain_google_genai', 'langchain_openai', 'pyngrok']:
    try: __import__(pkg.replace('-','_')); print(f'  ✓ {pkg}')
    except ImportError: print(f'  ✗ {pkg} MISSING')

In [ ]:
#@title 🦙 Cell 3 — Install Ollama + pull Qwen3-14B
#@markdown First run: ~5 min download. After that: instant from Drive cache.

import subprocess, os, shutil, time

DRIVE_DIR   = '/content/drive/MyDrive/SelfReplicatingAgent'
MODEL_CACHE = f'{DRIVE_DIR}/ollama_models'
OLLAMA_MODEL = 'llama4:scout'  #@param ["qwen3:14b", "qwen3:8b", "qwen3:30b-a3b", "qwen2.5-coder:14b"]

os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['OLLAMA_MODELS'] = MODEL_CACHE

if not shutil.which('ollama'):
    print('Installing Ollama...')
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
    print('✅ Ollama installed')
else:
    print('✅ Ollama already installed')

manifest = os.path.join(MODEL_CACHE, 'manifests', 'registry.ollama.ai',
                         'library', OLLAMA_MODEL.replace(':', '/'))
if os.path.exists(manifest):
    print(f'✅ {OLLAMA_MODEL} cached in Drive — no download needed')
else:
    sizes = {'qwen3:14b':'~9 GB','qwen3:8b':'~5 GB','qwen3:30b-a3b':'~20 GB','qwen2.5-coder:14b':'~9 GB'}
    print(f'⬇️  Downloading {OLLAMA_MODEL} ({sizes.get(OLLAMA_MODEL,"?")}). Please wait...')
    srv = subprocess.Popen(['ollama','serve'],
        env={**os.environ,'OLLAMA_MODELS':MODEL_CACHE},
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    r = subprocess.run(['ollama','pull',OLLAMA_MODEL],
        env={**os.environ,'OLLAMA_MODELS':MODEL_CACHE}, capture_output=True, text=True)
    srv.terminate()
    print('✅ Downloaded and cached' if r.returncode == 0 else f'❌ {r.stderr[-500:]}')

open('/content/ollama_model.txt','w').write(OLLAMA_MODEL)
print(f'📝 Model: {OLLAMA_MODEL}')

In [ ]:
#@title 🚀 Cell 4 — Start Ollama server

import subprocess, os, time, urllib.request, json

DRIVE_DIR   = '/content/drive/MyDrive/SelfReplicatingAgent'
MODEL_CACHE = f'{DRIVE_DIR}/ollama_models'
OLLAMA_MODEL = open('/content/ollama_model.txt').read().strip()

subprocess.run(['pkill','-f','ollama'], capture_output=True)
time.sleep(2)

subprocess.Popen(['ollama','serve'],
    env={**os.environ,'OLLAMA_MODELS':MODEL_CACHE},
    stdout=open('/content/ollama.log','w'), stderr=subprocess.STDOUT)

print('Waiting for Ollama...')
for i in range(25):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2)
        print(f'✅ Ollama ready ({i+1}s)')
        break
    except: time.sleep(1); print(f'  {i+1}/25', end='\r')
else:
    print('❌ Failed:', open('/content/ollama.log').read()[-1000:])

r = subprocess.run(['ollama','list'], capture_output=True, text=True,
    env={**os.environ,'OLLAMA_MODELS':MODEL_CACHE})
print(r.stdout)

os.environ['OLLAMA_BASE_URL'] = 'http://localhost:11434'
os.environ['OLLAMA_MODEL']    = OLLAMA_MODEL
print(f'🎯 {OLLAMA_MODEL} @ localhost:11434')

# Quick smoke test
try:
    req = urllib.request.Request('http://localhost:11434/api/generate',
        data=json.dumps({'model':OLLAMA_MODEL,'prompt':'/no_think Say OK','stream':False}).encode(),
        headers={'Content-Type':'application/json'})
    resp = json.loads(urllib.request.urlopen(req, timeout=90).read())
    print(f'🧪 {resp["response"].strip()[:100]}')
    print('✅ Working!')
except Exception as e:
    print(f'⚠️  Smoke test: {e}')

In [ ]:
#@title 🔑 Cell 5 — API keys (optional fallback)

import os
from google.colab import userdata

def _s(k):
    try: return userdata.get(k) or ''
    except: return ''

for k, v in [('GROQ_API_KEY', _s('GROQ_API_KEY')),
             ('GEMINI_API_KEY', _s('GEMINI_API_KEY'))]:
    if v: os.environ[k] = v; print(f'✅ {k} loaded')

p = ('🦙 Ollama qwen3:14b — no rate limits' if os.environ.get('OLLAMA_BASE_URL') else
     '⚡ Groq' if os.environ.get('GROQ_API_KEY') else
     '✨ Gemini' if os.environ.get('GEMINI_API_KEY') else '❌ No provider!')
print(f'Provider: {p}')

In [ ]:
#@title 🌐 Cell 9 — Start Web Dashboard (MISSION CONTROL)
#@markdown Starts serve.py + exposes it via ngrok → gives you a live public URL.
#@markdown Open the URL to see MISSION_CONTROL in real-time while evolution runs.
#@markdown
#@markdown **Optional:** Add your ngrok token for persistent URLs (free at ngrok.com).

import subprocess, sys, os, time, threading
from pathlib import Path

WORK_DIR     = '/content/SelfReplicatingAgent'
NGROK_TOKEN  = ''  #@param {type:"string"}
#@markdown Leave NGROK_TOKEN blank to use anonymous tunnel (changes each session).

# Set PORT to 7860 (HF-compatible)
os.environ['PORT'] = '7860'

# ── Start serve.py (HTTP server — does NOT auto-launch evolution) ────────────
# We pass --no-evolution so serve.py only serves the dashboard;
# Cell 6 will launch evolution separately.
serve_proc = subprocess.Popen(
    [sys.executable, 'serve.py'],
    cwd=WORK_DIR,
    env={**os.environ, 'SERVE_ONLY': '1'},  # flag to suppress auto-evolution start
    stdout=open('/content/serve.log', 'w', buffering=1),
    stderr=subprocess.STDOUT
)
print(f'▶ serve.py PID={serve_proc.pid}')
time.sleep(2)

# ── Start ngrok tunnel ───────────────────────────────────────────────────────
from pyngrok import ngrok, conf

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
    print('✅ ngrok authenticated')

# Kill any existing tunnels
ngrok.kill()
time.sleep(1)

tunnel = ngrok.connect(7860, bind_tls=True)
public_url = tunnel.public_url

print()
print('=' * 60)
print('🖥️  MISSION CONTROL DASHBOARD')
print('=' * 60)
print(f'🔗  {public_url}/MISSION_CONTROL.html')
print()
print('Open the link above in your browser.')
print('The dashboard updates every 3 seconds automatically.')
print('=' * 60)
print()
print('Next: Run Cell 6 to start the evolution pipeline.')
print('The dashboard will show live progress as tasks complete.')

# Save URL for reference
open('/content/dashboard_url.txt', 'w').write(f'{public_url}/MISSION_CONTROL.html')

# Monitor serve.py (print any startup errors)
def _tail_log():
    import time
    time.sleep(3)
    try:
        log = open('/content/serve.log').read()
        if 'Error' in log or 'error' in log:
            print('\n⚠️  serve.py log:\n', log[-500:])
    except: pass
threading.Thread(target=_tail_log, daemon=True).start()

In [ ]:
#@title 🧬 Cell 6 — Run Evolution Pipeline
#@markdown Run AFTER Cell 9. Watch progress at the dashboard URL above.

import subprocess, sys, os, json
from pathlib import Path

WORK_DIR  = '/content/SelfReplicatingAgent'
DATA_DIR  = f'{WORK_DIR}/data'
START_GEN = 1  #@param {type:"integer"}
MAX_GENS  = 10 #@param {type:"integer"}

os.makedirs(DATA_DIR, exist_ok=True)

# Show dashboard URL reminder
url_file = Path('/content/dashboard_url.txt')
if url_file.exists():
    print(f'📺 Dashboard: {url_file.read_text().strip()}')
    print()

# Clear stop flag
stop_flag = Path(f'{DATA_DIR}/stop.flag')
if stop_flag.exists(): stop_flag.unlink(); print('🗑️  Cleared stop.flag')

print(f'🚀 Gen {START_GEN} → Gen {START_GEN + MAX_GENS - 1}')
print(f'🦙 {os.environ.get("OLLAMA_MODEL", "API fallback")}')
print(f'⏱️  ~{MAX_GENS * 2} hrs total, ~{MAX_GENS * 4} Colab units')
print('=' * 60)

gen_dir = Path(WORK_DIR) / 'generations' / f'gen_{START_GEN}'
if not gen_dir.exists():
    raise FileNotFoundError(f'{gen_dir}')

log_file = open(f'{DATA_DIR}/colab_run.log', 'w', buffering=1)

proc = subprocess.Popen(
    [sys.executable, 'main.py', '--gen', str(START_GEN)],
    cwd=str(gen_dir), env={**os.environ},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
print(f'▶ PID={proc.pid}')

try:
    for line in proc.stdout:
        line = line.rstrip()
        if line: print(line); log_file.write(line + '\n')
except KeyboardInterrupt:
    print('\n⏹ Stopped'); proc.terminate()
finally:
    log_file.close()

ret = proc.wait()
print(f'\n✅ Done (exit {ret})')

results_dir = gen_dir / 'results'
if results_dir.exists():
    passed = sum(1 for f in results_dir.glob('*.json')
                 if json.loads(f.read_text()).get('status') == 'success')
    total = len(list(results_dir.glob('*.json')))
    if total:
        bar = '█'*passed + '░'*(total-passed)
        print(f'📊 [{bar[:20]}] {passed/total*100:.1f}% ({passed}/{total})')

# Remind of dashboard URL
if url_file.exists():
    print(f'\n📺 Dashboard: {url_file.read_text().strip()}')

In [ ]:
#@title 📊 Cell 7 — Results Summary

import json
from pathlib import Path

WORK_DIR = '/content/SelfReplicatingAgent'
gens_dir = Path(WORK_DIR) / 'generations'

# Show dashboard URL
url_f = Path('/content/dashboard_url.txt')
if url_f.exists(): print(f'📺 Dashboard: {url_f.read_text().strip()}\n')

print('='*60)
print('  EVOLUTION RESULTS')
print('='*60)

total_p = total_f = 0
for gen_dir in sorted(gens_dir.iterdir()):
    if not gen_dir.name.startswith('gen_'): continue
    rdir = gen_dir / 'results'
    if not rdir.exists(): continue

    p = f = 0
    errs = {}
    for rf in rdir.glob('*.json'):
        try:
            d = json.loads(rf.read_text())
            if d.get('status') == 'success': p += 1
            else:
                f += 1
                e = str(d.get('error_type', d.get('error','?')))[:35]
                errs[e] = errs.get(e,0)+1
        except: pass

    tot = p+f
    if not tot: continue
    pct = p/tot*100
    bar = '█'*p + '░'*f
    icon = '📈' if pct>=50 else ('📉' if pct<25 else '➡️')
    print(f'\n{icon} {gen_dir.name.upper():8s} [{bar[:20]:20s}] {pct:5.1f}% ({p}/{tot})')
    for e,c in sorted(errs.items(), key=lambda x:-x[1])[:3]:
        print(f'  └─ {e}: {c}x')
    total_p+=p; total_f+=f

if total_p+total_f:
    print(f'\n{"-"*60}')
    print(f'Overall: {total_p/(total_p+total_f)*100:.1f}% ({total_p}/{total_p+total_f})')